# Notebook 10 - Model Serving REST API
**Project**: Real-Time Retail Analytics and Demand Prediction Platform

**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar

Deploys the trained ML model as a REST API accepting POST requests.

API URL: **http://localhost:3001**

## 10.1 Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install flask joblib scikit-learn pandas numpy requests -q
print('Done.')

Done.


## 10.2 Load Trained Model

In [2]:
import joblib
import pandas as pd
import numpy as np
import os

MODEL_PATH = '/home/jovyan/work/model_bundle.joblib'

bundle = joblib.load(MODEL_PATH)
model = bundle['model']
features = bundle['features']
model_name = bundle['model_name']

print(f'Model loaded: {model_name}')
print(f'Features ({len(features)}): {features}')

# Quick test
test_input = pd.DataFrame([{
    'Year': 2010, 'Month': 11, 'WeekOfYear': 46, 'DayOfWeek': 3,
    'AvgUnitPrice': 3.75, 'NumInvoices': 5, 'UniqueCustomers': 4,
    'TotalRevenue': 185.0, 'StockCode_idx': 42, 'Country_idx': 0
}])
pred = model.predict(test_input[features])
print(f'Test prediction: {pred[0]:.1f} units')

Model loaded: Linear Regression
Features (10): ['Year', 'Month', 'WeekOfYear', 'DayOfWeek', 'AvgUnitPrice', 'NumInvoices', 'UniqueCustomers', 'TotalRevenue', 'StockCode_idx', 'Country_idx']
Test prediction: 123.3 units


## 10.3 Create and Start REST API Server

In [3]:
from flask import Flask, request, jsonify
import threading
import time

FEATURES = features

api = Flask('retail_demand_api')

@api.route('/', methods=['GET'])
def home():
    return '''
    <html>
    <head>
        <title>Retail Demand Prediction API</title>
        <style>
            * { margin: 0; padding: 0; box-sizing: border-box; }
            body { background: #080b12; color: #e2e8f8; font-family: 'Segoe UI', sans-serif; min-height: 100vh; display: flex; justify-content: center; align-items: center; }
            .container { background: #111827; border: 1px solid #1f2b47; border-radius: 16px; padding: 32px; width: 500px; }
            h1 { color: #3b82f6; font-size: 22px; margin-bottom: 4px; }
            .sub { color: #6b7fa3; font-size: 12px; margin-bottom: 24px; }
            .status { background: #10b981; color: white; padding: 4px 12px; border-radius: 20px; font-size: 11px; font-weight: 700; display: inline-block; margin-bottom: 20px; }
            label { color: #6b7fa3; font-size: 11px; display: block; margin-bottom: 4px; margin-top: 12px; text-transform: uppercase; letter-spacing: 1px; }
            input, select { width: 100%; padding: 10px; background: #080b12; color: #e2e8f8; border: 1px solid #1f2b47; border-radius: 8px; font-size: 14px; }
            .row { display: flex; gap: 12px; }
            .row > div { flex: 1; }
            button { width: 100%; padding: 12px; background: #3b82f6; color: white; border: none; border-radius: 8px; font-size: 16px; font-weight: 700; cursor: pointer; margin-top: 20px; }
            button:hover { background: #2563eb; }
            .result { background: #080b12; border: 1px solid #1f2b47; border-radius: 8px; padding: 16px; margin-top: 16px; text-align: center; min-height: 60px; }
            .qty { color: #10b981; font-size: 32px; font-weight: 800; }
            .info { color: #6b7fa3; font-size: 11px; margin-top: 4px; }
            .endpoints { background: #0c1221; border-radius: 8px; padding: 12px; margin-top: 20px; }
            .endpoints h3 { color: #3b82f6; font-size: 13px; margin-bottom: 8px; }
            .ep { color: #6b7fa3; font-size: 11px; margin: 4px 0; }
            .ep span { color: #10b981; font-weight: 700; }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Retail Demand Prediction API</h1>
            <div class="sub">IIT Madras Zanzibar | Z5008 | Vineet Joshi | ZDA25M007</div>
            <div class="status">API LIVE</div>
            <div class="row">
                <div><label>Month</label><select id="month">
                    <option value="1">January</option><option value="2">February</option><option value="3">March</option><option value="4">April</option><option value="5">May</option><option value="6">June</option><option value="7">July</option><option value="8">August</option><option value="9">September</option><option value="10">October</option><option value="11" selected>November</option><option value="12">December</option>
                </select></div>
                <div><label>Day of Week</label><select id="dow">
                    <option value="1">Monday</option><option value="2">Tuesday</option><option value="3" selected>Wednesday</option><option value="4">Thursday</option><option value="5">Friday</option><option value="6">Saturday</option><option value="7">Sunday</option>
                </select></div>
            </div>
            <div class="row">
                <div><label>Avg Unit Price</label><input type="number" id="price" value="3.75" step="0.25" min="0.1"></div>
                <div><label>Invoices</label><input type="number" id="inv" value="5" step="1" min="1"></div>
            </div>
            <div class="row">
                <div><label>Customers</label><input type="number" id="cust" value="4" step="1" min="1"></div>
                <div><label>Revenue</label><input type="number" id="rev" value="185" step="10" min="0"></div>
            </div>
            <button onclick="predict()">Predict Demand</button>
            <div class="result" id="result"><div style="color:#6b7fa3">Click Predict to see result</div></div>
            <div class="endpoints"><h3>API Endpoints</h3>
                <div class="ep"><span>POST</span> /predict</div>
                <div class="ep"><span>POST</span> /batch_predict</div>
                <div class="ep"><span>GET</span> /health</div>
                <div class="ep"><span>GET</span> /model_info</div>
            </div>
        </div>
        <script>
        async function predict() {
            const p = {Year:2010, Month:parseInt(document.getElementById('month').value), WeekOfYear:parseInt(document.getElementById('month').value)*4, DayOfWeek:parseInt(document.getElementById('dow').value), AvgUnitPrice:parseFloat(document.getElementById('price').value), NumInvoices:parseInt(document.getElementById('inv').value), UniqueCustomers:parseInt(document.getElementById('cust').value), TotalRevenue:parseFloat(document.getElementById('rev').value), StockCode_idx:50, Country_idx:0};
            try {
                const r = await fetch('/predict', {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(p)});
                const d = await r.json();
                const m = parseInt(document.getElementById('month').value);
                const s = (m===11||m===12)?'Peak Season':(m>=6&&m<=8)?'Summer':'Normal';
                document.getElementById('result').innerHTML = '<div class="qty">'+d.predicted_quantity.toLocaleString()+' units</div><div class="info">'+s+' | Model: '+d.model+'</div>';
            } catch(e) { document.getElementById('result').innerHTML = '<div style="color:#ef4444">Error</div>'; }
        }
        </script>
    </body></html>'''

@api.route('/predict', methods=['POST'])
def predict_api():
    data = request.get_json()
    df = pd.DataFrame([data])
    for f in FEATURES:
        if f not in df.columns:
            df[f] = 0
    pred = model.predict(df[FEATURES])
    qty = float(np.clip(pred[0], 0, None))
    return jsonify({'predicted_quantity': round(qty, 2), 'model': model_name, 'status': 'ok'})

@api.route('/batch_predict', methods=['POST'])
def batch_predict():
    data = request.get_json()
    df = pd.DataFrame(data)
    for f in FEATURES:
        if f not in df.columns:
            df[f] = 0
    preds = model.predict(df[FEATURES])
    return jsonify([{'predicted_quantity': round(float(np.clip(p, 0, None)), 2)} for p in preds])

@api.route('/health', methods=['GET', 'POST'])
def health():
    return jsonify({'status': 'healthy', 'model': model_name, 'service': 'retail_demand_api', 'features': FEATURES})

@api.route('/model_info', methods=['GET'])
def model_info():
    return jsonify({'model_name': model_name, 'features': FEATURES, 'target': 'TotalQuantity', 'project': 'Z5008 Retail Analytics', 'author': 'Vineet Joshi | ZDA25M007'})

def run_api():
    api.run(host='0.0.0.0', port=3001, debug=False, use_reloader=False)

thread = threading.Thread(target=run_api, daemon=True)
thread.start()
time.sleep(3)

print('API running at http://localhost:3001')
print('Web UI: http://localhost:3001/')
print('Endpoints: /predict, /batch_predict, /health, /model_info')

 * Serving Flask app 'retail_demand_api'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:3001
 * Running on http://172.18.0.10:3001
Press CTRL+C to quit


API running at http://localhost:3001
Web UI: http://localhost:3001/
Endpoints: /predict, /batch_predict, /health, /model_info


127.0.0.1 - - [28/Apr/2026 21:05:12] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:12] "GET /model_info HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:16] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:20] "POST /batch_predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:24] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 21:05:29] "POST

## 10.4 Test API - Health Check

In [4]:
import requests
import json

API = 'http://localhost:3001'

print('GET /health')
resp = requests.get(f'{API}/health', timeout=5)
print(f'Status: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

print('\nGET /model_info')
resp = requests.get(f'{API}/model_info', timeout=5)
print(f'Status: {resp.status_code}')
print(json.dumps(resp.json(), indent=2))

GET /health
Status: 200
{
  "features": [
    "Year",
    "Month",
    "WeekOfYear",
    "DayOfWeek",
    "AvgUnitPrice",
    "NumInvoices",
    "UniqueCustomers",
    "TotalRevenue",
    "StockCode_idx",
    "Country_idx"
  ],
  "model": "Linear Regression",
  "service": "retail_demand_api",
  "status": "healthy"
}

GET /model_info
Status: 200
{
  "author": "Vineet Joshi | ZDA25M007",
  "features": [
    "Year",
    "Month",
    "WeekOfYear",
    "DayOfWeek",
    "AvgUnitPrice",
    "NumInvoices",
    "UniqueCustomers",
    "TotalRevenue",
    "StockCode_idx",
    "Country_idx"
  ],
  "model_name": "Linear Regression",
  "project": "Z5008 Retail Analytics",
  "target": "TotalQuantity"
}


## 10.5 Test API - Single Prediction (POST)

In [5]:
payload = {
    'Year': 2010,
    'Month': 12,
    'WeekOfYear': 50,
    'DayOfWeek': 5,
    'AvgUnitPrice': 5.50,
    'NumInvoices': 8,
    'UniqueCustomers': 6,
    'TotalRevenue': 440.0,
    'StockCode_idx': 100,
    'Country_idx': 0
}

print('POST /predict')
print(f'Request: {json.dumps(payload, indent=2)}')
print()

resp = requests.post(f'{API}/predict', json=payload, timeout=5)
print(f'Status: {resp.status_code}')
print(f'Response: {json.dumps(resp.json(), indent=2)}')

POST /predict
Request: {
  "Year": 2010,
  "Month": 12,
  "WeekOfYear": 50,
  "DayOfWeek": 5,
  "AvgUnitPrice": 5.5,
  "NumInvoices": 8,
  "UniqueCustomers": 6,
  "TotalRevenue": 440.0,
  "StockCode_idx": 100,
  "Country_idx": 0
}

Status: 200
Response: {
  "model": "Linear Regression",
  "predicted_quantity": 300.23,
  "status": "ok"
}


## 10.6 Test API - Batch Prediction (POST)

In [6]:
batch_payload = [
    {'Year':2010,'Month':1,'WeekOfYear':4,'DayOfWeek':2,'AvgUnitPrice':2.5,'NumInvoices':3,'UniqueCustomers':3,'TotalRevenue':75.0,'StockCode_idx':20,'Country_idx':0},
    {'Year':2010,'Month':6,'WeekOfYear':24,'DayOfWeek':4,'AvgUnitPrice':8.0,'NumInvoices':12,'UniqueCustomers':10,'TotalRevenue':960.0,'StockCode_idx':55,'Country_idx':0},
    {'Year':2010,'Month':11,'WeekOfYear':46,'DayOfWeek':3,'AvgUnitPrice':3.75,'NumInvoices':5,'UniqueCustomers':4,'TotalRevenue':185.0,'StockCode_idx':42,'Country_idx':0},
    {'Year':2010,'Month':12,'WeekOfYear':51,'DayOfWeek':1,'AvgUnitPrice':1.50,'NumInvoices':20,'UniqueCustomers':15,'TotalRevenue':300.0,'StockCode_idx':80,'Country_idx':5},
    {'Year':2010,'Month':3,'WeekOfYear':12,'DayOfWeek':6,'AvgUnitPrice':15.0,'NumInvoices':2,'UniqueCustomers':2,'TotalRevenue':60.0,'StockCode_idx':10,'Country_idx':2},
]

print(f'POST /batch_predict ({len(batch_payload)} items)')
resp = requests.post(f'{API}/batch_predict', json=batch_payload, timeout=5)
print(f'Status: {resp.status_code}')
print()

results = resp.json()
months = ['', 'Jan','','Mar','','','Jun','','','','','Nov','Dec']
for i, (r, p) in enumerate(zip(results, batch_payload)):
    m = p['Month']
    print(f"  Scenario {i+1}: Month={m:>2} Price={p['AvgUnitPrice']:>5.1f} Invoices={p['NumInvoices']:>2}  -->  {r['predicted_quantity']:>8.1f} units")

POST /batch_predict (5 items)
Status: 200

  Scenario 1: Month= 1 Price=  2.5 Invoices= 3  -->      53.5 units
  Scenario 2: Month= 6 Price=  8.0 Invoices=12  -->     687.8 units
  Scenario 3: Month=11 Price=  3.8 Invoices= 5  -->     123.3 units
  Scenario 4: Month=12 Price=  1.5 Invoices=20  -->      39.5 units
  Scenario 5: Month= 3 Price= 15.0 Invoices= 2  -->      26.2 units


## 10.7 Load Test (10x) - System Stability

In [7]:
test_payload = {
    'Year':2010,'Month':11,'WeekOfYear':46,'DayOfWeek':3,
    'AvgUnitPrice':3.75,'NumInvoices':5,'UniqueCustomers':4,
    'TotalRevenue':185.0,'StockCode_idx':42,'Country_idx':0
}

N = 10
print(f'Running {N}x load test on POST /predict...')
print()

times = []
successes = 0

for i in range(N):
    start = time.time()
    resp = requests.post(f'{API}/predict', json=test_payload, timeout=10)
    elapsed = (time.time() - start) * 1000
    times.append(elapsed)
    if resp.status_code == 200:
        successes += 1
    qty = resp.json().get('predicted_quantity', '?')
    print(f'  Request {i+1:>2}/{N} | {"OK" if resp.status_code==200 else "FAIL"} | {elapsed:>6.1f}ms | Qty: {qty}')

print()
print('='*50)
print('LOAD TEST RESULTS')
print('='*50)
print(f'  Total requests  : {N}')
print(f'  Successful      : {successes}/{N} ({successes/N*100:.0f}%)')
print(f'  Avg latency     : {np.mean(times):.1f}ms')
print(f'  Min latency     : {np.min(times):.1f}ms')
print(f'  Max latency     : {np.max(times):.1f}ms')
print(f'  P95 latency     : {np.percentile(times, 95):.1f}ms')
print(f'  Throughput      : {N/(sum(times)/1000):.1f} req/sec')

Running 10x load test on POST /predict...

  Request  1/10 | OK |   19.1ms | Qty: 123.31
  Request  2/10 | OK |   12.4ms | Qty: 123.31
  Request  3/10 | OK |   12.4ms | Qty: 123.31
  Request  4/10 | OK |   22.5ms | Qty: 123.31
  Request  5/10 | OK |   21.4ms | Qty: 123.31
  Request  6/10 | OK |   17.0ms | Qty: 123.31
  Request  7/10 | OK |   58.9ms | Qty: 123.31
  Request  8/10 | OK |   19.4ms | Qty: 123.31
  Request  9/10 | OK |   22.2ms | Qty: 123.31
  Request 10/10 | OK |   14.5ms | Qty: 123.31

LOAD TEST RESULTS
  Total requests  : 10
  Successful      : 10/10 (100%)
  Avg latency     : 22.0ms
  Min latency     : 12.4ms
  Max latency     : 58.9ms
  P95 latency     : 42.5ms
  Throughput      : 45.5 req/sec


## 10.8 Extended Load Test (100x)

In [8]:
N2 = 100
print(f'Running {N2}x extended load test...')

times2 = []
errors = 0
start_all = time.time()

for i in range(N2):
    # Vary the payload slightly each time
    p = test_payload.copy()
    p['Month'] = np.random.randint(1, 13)
    p['AvgUnitPrice'] = round(np.random.uniform(0.5, 20.0), 2)
    p['NumInvoices'] = np.random.randint(1, 25)
    
    start = time.time()
    try:
        resp = requests.post(f'{API}/predict', json=p, timeout=10)
        elapsed = (time.time() - start) * 1000
        times2.append(elapsed)
    except:
        errors += 1
    
    if (i+1) % 25 == 0:
        print(f'  Completed {i+1}/{N2} requests...')

total_time = time.time() - start_all

print()
print('='*50)
print('EXTENDED LOAD TEST RESULTS')
print('='*50)
print(f'  Total requests  : {N2}')
print(f'  Errors          : {errors}')
print(f'  Success rate    : {(N2-errors)/N2*100:.1f}%')
print(f'  Total time      : {total_time:.2f}s')
print(f'  Avg latency     : {np.mean(times2):.1f}ms')
print(f'  P50 latency     : {np.percentile(times2, 50):.1f}ms')
print(f'  P95 latency     : {np.percentile(times2, 95):.1f}ms')
print(f'  P99 latency     : {np.percentile(times2, 99):.1f}ms')
print(f'  Throughput      : {N2/total_time:.1f} req/sec')
print(f'  System status   : STABLE')

Running 100x extended load test...
  Completed 25/100 requests...
  Completed 50/100 requests...
  Completed 75/100 requests...
  Completed 100/100 requests...

EXTENDED LOAD TEST RESULTS
  Total requests  : 100
  Errors          : 0
  Success rate    : 100.0%
  Total time      : 1.20s
  Avg latency     : 11.8ms
  P50 latency     : 11.5ms
  P95 latency     : 15.2ms
  P99 latency     : 18.2ms
  Throughput      : 83.5 req/sec
  System status   : STABLE


## 10.9 Create Dockerfile

In [9]:
# Create standalone service file
service_code = '''from flask import Flask, request, jsonify
import joblib
import pandas as pd
import numpy as np
import os

# Load model
bundle = joblib.load('model_bundle.joblib')
model = bundle['model']
features = bundle['features']
model_name = bundle['model_name']

app = Flask('retail_demand_api')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    df = pd.DataFrame([data])
    for f in features:
        if f not in df.columns:
            df[f] = 0
    pred = model.predict(df[features])
    qty = float(np.clip(pred[0], 0, None))
    return jsonify({'predicted_quantity': round(qty, 2), 'model': model_name, 'status': 'ok'})

@app.route('/batch_predict', methods=['POST'])
def batch_predict():
    data = request.get_json()
    df = pd.DataFrame(data)
    for f in features:
        if f not in df.columns:
            df[f] = 0
    preds = model.predict(df[features])
    return jsonify([{'predicted_quantity': round(float(np.clip(p, 0, None)), 2)} for p in preds])

@app.route('/health', methods=['GET', 'POST'])
def health():
    return jsonify({'status': 'healthy', 'model': model_name, 'service': 'retail_demand_api'})

@app.route('/model_info', methods=['GET'])
def model_info():
    return jsonify({'model_name': model_name, 'features': features, 'target': 'TotalQuantity'})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=3001)
'''

with open('/home/jovyan/work/retail_api.py', 'w') as f:
    f.write(service_code)

# Create Dockerfile
dockerfile = '''FROM python:3.11-slim
WORKDIR /app
COPY model_bundle.joblib .
COPY retail_api.py .
RUN pip install flask scikit-learn pandas numpy joblib
EXPOSE 3001
CMD ["python", "retail_api.py"]
'''

with open('/home/jovyan/work/Dockerfile', 'w') as f:
    f.write(dockerfile)

# Create requirements.txt
with open('/home/jovyan/work/requirements.txt', 'w') as f:
    f.write('flask\nscikit-learn\npandas\nnumpy\njoblib\n')

print('Files created:')
print('  retail_api.py      - standalone API server')
print('  Dockerfile         - Docker build file')
print('  requirements.txt   - Python dependencies')
print()
print('To build and run Docker container:')
print('  docker build -t retail-api .')
print('  docker run -p 3001:3001 retail-api')

Files created:
  retail_api.py      - standalone API server
  Dockerfile         - Docker build file
  requirements.txt   - Python dependencies

To build and run Docker container:
  docker build -t retail-api .
  docker run -p 3001:3001 retail-api


## 10.10 API Documentation

In [10]:
print('='*60)
print('REST API DOCUMENTATION')
print('='*60)
print(f'\nBase URL: http://localhost:3001')
print()
print('ENDPOINTS:')
print('-'*60)
print()
print('1. POST /predict')
print('   Single demand prediction')
print('   Input:  JSON object with features')
print('   Output: {"predicted_quantity": 125.0, "model": "...", "status": "ok"}')
print()
print('2. POST /batch_predict')
print('   Batch predictions')
print('   Input:  JSON array of objects')
print('   Output: [{"predicted_quantity": 125.0}, ...]')
print()
print('3. GET /health')
print('   Health check')
print('   Output: {"status": "healthy", "model": "...", "service": "..."}')
print()
print('4. GET /model_info')
print('   Model metadata')
print('   Output: {"model_name": "...", "features": [...], "target": "..."}')
print()
print('CURL EXAMPLES:')
print('-'*60)
print()
print('curl http://localhost:3001/health')
print()
print('curl -X POST http://localhost:3001/predict \\')
print('  -H "Content-Type: application/json" \\')
print('  -d \'{"Month":11,"DayOfWeek":3,"AvgUnitPrice":3.75,"NumInvoices":5,"UniqueCustomers":4,"TotalRevenue":185,"StockCode_idx":42,"Country_idx":0,"Year":2010,"WeekOfYear":46}\'')

REST API DOCUMENTATION

Base URL: http://localhost:3001

ENDPOINTS:
------------------------------------------------------------

1. POST /predict
   Single demand prediction
   Input:  JSON object with features
   Output: {"predicted_quantity": 125.0, "model": "...", "status": "ok"}

2. POST /batch_predict
   Batch predictions
   Input:  JSON array of objects
   Output: [{"predicted_quantity": 125.0}, ...]

3. GET /health
   Health check
   Output: {"status": "healthy", "model": "...", "service": "..."}

4. GET /model_info
   Model metadata
   Output: {"model_name": "...", "features": [...], "target": "..."}

CURL EXAMPLES:
------------------------------------------------------------

curl http://localhost:3001/health

curl -X POST http://localhost:3001/predict \
  -H "Content-Type: application/json" \
  -d '{"Month":11,"DayOfWeek":3,"AvgUnitPrice":3.75,"NumInvoices":5,"UniqueCustomers":4,"TotalRevenue":185,"StockCode_idx":42,"Country_idx":0,"Year":2010,"WeekOfYear":46}'


## 10.11 Summary

In [11]:
print('='*55)
print('NOTEBOOK 10 COMPLETE')
print('='*55)
print(f'  Model        : {model_name}')
print(f'  API URL      : http://localhost:3001')
print(f'  Endpoints    : /predict, /batch_predict, /health, /model_info')
print(f'  10x load test: PASSED')
print(f'  100x load test: PASSED')
print()
print('Files created:')
print('  retail_api.py')
print('  Dockerfile')
print('  requirements.txt')
print()
print('Docker commands:')
print('  docker build -t retail-api /home/jovyan/work/')
print('  docker run -p 3001:3001 retail-api')

NOTEBOOK 10 COMPLETE
  Model        : Linear Regression
  API URL      : http://localhost:3001
  Endpoints    : /predict, /batch_predict, /health, /model_info
  10x load test: PASSED
  100x load test: PASSED

Files created:
  retail_api.py
  Dockerfile
  requirements.txt

Docker commands:
  docker build -t retail-api /home/jovyan/work/
  docker run -p 3001:3001 retail-api
